A neural network for Predicting How Well Prospective Drugs Bind To A Protein<br>
Authors: Rebecka Antonsson and Ellen Nilsson<br>
Latest update: 30-10-2025

# Aim
The aim of this project is to construct a basic model that can predict the binding affinity of potential drugs to a cell surface receptor. For this model only one receptor is chosen, Epidermal growth factor receptor (EGFR). EGFR is often mutated or over expressed in many cancers, and therefore many drugs are designed to bind to it. To determine if a molecule will bind to EGFR or not, a model is trained with features of the molecues and what the binding affinity of these molecules are.

Most of the information about the molecules is given from so called Fingerprints, they are fixed-length binary vectors representing the presence of certain sub-structures in a molecule as well as  its shape, size and which atoms are contained. Other information included is how well they bind to EGFR and other chemical properties such as molecular weight.

From the information about the molecules the purpose of this project is then to predict the pCHEMBL value which can be described as "how well the molecule binds to the EGFR". To create precitions of pCHEMBL values, a Neural Network has been created. It can be argued that something as advanced as a Neural Network is not actually needed for this quite simple problem. However, the idea of this project is to create a model which can be bulit upon. The model should be able to expand to include several different receptors, and more diverse molecules.


# Load libraries

In [ ]:
# Load libraries

if 1:
  import pandas as pd
  from google.colab import files
  import io
  import matplotlib.pyplot as plt
  import numpy as np
  from sklearn.model_selection import train_test_split

  import keras
  from keras.models import Sequential, Model
  from keras.layers import Dense, Dropout, Flatten, Normalization, Conv2D, MaxPooling2D, LeakyReLU
  from keras.callbacks import EarlyStopping
  from sklearn.model_selection import train_test_split
  from keras.utils import to_categorical

  # Load rdKit
  !pip install rdkit
  from rdkit import Chem
  from rdkit.Chem import AllChem, Draw, rdFingerprintGenerator

# Input data

### Finding the input data
1. Go to the CHEMBL website https://www.ebi.ac.uk/chembl/
2. Search for the target molecule (EGFR), with ID CHEMBL203 and choose it in the list
3. Click on "Activity charts" or "Bioactivites" or similar in the menu to the left.
4. Click on the "Activity Types for CHEMBL203 (Epidermal growth factor receptor)"
5. Filter so that you only have the molecules with IC50 values
6. In the top right klick on "csv", let it load and download the file with molecules with IC50 values.

### Description of the input data
The data is downloaded from the CHEMBL webiste. The CHEMBL website is a public database of bioactive molecules with measured activities.
The molecule EFGR was selected, and then a file containing molecules with known binding affinity, or lack thereof, to EFGR was downloaded. First molecules that had IC50 data was filtered, this is to make the molecules binding strength comparable to oneanother.

In [ ]:
# Load data
# Select the CHEMBL EGFR csv input file that you downloaded according to the instructions above
uploaded = files.upload()

Saving DOWNLOAD-gU8RPQ5Wut7KaKJdHzr2fUYYJcpIjb0ClUND2cUakNk_eq_.csv to DOWNLOAD-gU8RPQ5Wut7KaKJdHzr2fUYYJcpIjb0ClUND2cUakNk_eq_.csv


In [ ]:
# Make the loaded data into a pandas data frame
CHEMBL_df_OG = pd.read_csv(io.BytesIO(uploaded[list(uploaded.keys())[0]]), sep=';')

In [ ]:
# Print all the column names (check if data has been downloaded correctly)
list(CHEMBL_df_OG)

['Molecule ChEMBL ID',
 'Molecule Name',
 'Molecule Max Phase',
 'Molecular Weight',
 '#RO5 Violations',
 'AlogP',
 'Compound Key',
 'Smiles',
 'Standard Type',
 'Standard Relation',
 'Standard Value',
 'Standard Units',
 'pChEMBL Value',
 'Data Validity Comment',
 'Comment',
 'Uo Units',
 'Ligand Efficiency BEI',
 'Ligand Efficiency LE',
 'Ligand Efficiency LLE',
 'Ligand Efficiency SEI',
 'Potential Duplicate',
 'Assay ChEMBL ID',
 'Assay Description',
 'Assay Type',
 'BAO Format ID',
 'BAO Label',
 'Assay Organism',
 'Assay Tissue ChEMBL ID',
 'Assay Tissue Name',
 'Assay Cell Type',
 'Assay Subcellular Fraction',
 'Assay Parameters',
 'Assay Variant Accession',
 'Assay Variant Mutation',
 'Target ChEMBL ID',
 'Target Name',
 'Target Organism',
 'Target Type',
 'Document ChEMBL ID',
 'Source ID',
 'Source Description',
 'Document Journal',
 'Document Year',
 'Cell ChEMBL ID',
 'Properties',
 'Action Type',
 'Standard Text Value',
 'Value']

This model will focus on the binding strength. Below it is investigated what categories of binding strengths of the molecules is contained in our data. The column 'pChEMBL Value' is used, these values are a log scale of the experimentally measured IC50 values.

**IC50 is defined by [Wikipedia](https://en.wikipedia.org/wiki/IC50):** \
*"Half maximal inhibitory concentration (IC50) is a measure of the potency of a substance in inhibiting a specific biological or biochemical function. IC50 is a quantitative measure that indicates how much of a particular inhibitory substance (e.g. drug) is needed to inhibit, in vitro, a given biological process or biological component by 50%.[1] The biological component could be an enzyme, cell, cell receptor or microbe. IC50 values are typically expressed as molar concentration."*

In [ ]:
CHEMBL_df_OG['Standard Relation'].value_counts()

,count
Standard Relation,
'=',18988
'>',3215
'<',2056
'<=',87
'>>',11
'>=',8
'~',1


**The data contains a variety of standard relations with some only giving an approxiamte value or a magniude. A decision to only use the values that have an exact IC50, was made. This means only keeping the values "=" in the "Standard Relation" column (i.e keeping 18 988 data points).**

In [ ]:
# Clean up whitespace and quotes
CHEMBL_df_OG['Standard Relation'] = CHEMBL_df_OG['Standard Relation'].astype(str).str.strip().str.replace("'", "")

# Filter out "=" values from the standard relation column
CHEMBL_df_cleaned = CHEMBL_df_OG
CHEMBL_df_cleaned = CHEMBL_df_OG[CHEMBL_df_cleaned['Standard Relation'] == '=']

# Add column with categories "strong binders", "weak binders" and "non binders"
CHEMBL_df_classified = CHEMBL_df_cleaned.copy() # Create a copy to avoid SettingWithCopyWarning
CHEMBL_df_classified["Binding Category"] = ["Strong binders" if x >= 7 and x <= 9 else "Weak binders" if x >= 5 and x < 7 else "Non-binders" for x in CHEMBL_df_cleaned["pChEMBL Value"]]

In [ ]:
# Specify which columns to use in our model

CHEMBL_df_relevant = CHEMBL_df_classified[['Smiles', 'Binding Category' , 'Molecular Weight',
                       'AlogP','pChEMBL Value', 'Data Validity Comment']]

# Use onehotencoding on 'Data Validity Comment'
CHEMBL_df_encoded = pd.get_dummies(CHEMBL_df_relevant, columns=['Data Validity Comment'])

#filter
CHEMBL_df_encoded = CHEMBL_df_encoded[CHEMBL_df_encoded['Data Validity Comment_Outside typical range'] == 0]
CHEMBL_df_encoded = CHEMBL_df_encoded[CHEMBL_df_encoded['Data Validity Comment_Potential transcription error'] == 0]

# Create fingerprints for our molecules with RDKit


In [ ]:
# Function to convert SMILES into fingerprints
def smiles_to_fingerprint(smiles):
    if not isinstance(smiles, str):  # Check if the input is not a string
        return np.zeros((1024))
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:  # Handle cases where SMILES is invalid
        return np.zeros((1024))
    fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    fp = fp_gen.GetFingerprint(mol)
    arr = np.zeros((1,))
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

In [ ]:
# Create a new column for fingerprints
CHEMBL_df_encoded['Fingerprint'] = CHEMBL_df_encoded['Smiles'].apply(smiles_to_fingerprint)

# Flatten the fingerprint column into separate columns
fingerprint_df = pd.DataFrame(CHEMBL_df_encoded['Fingerprint'].tolist(), index=CHEMBL_df_encoded.index)

CHEMBL_df_processed = CHEMBL_df_encoded

# Make MLP model in Keras
Before building the model, columns are cleaned from NaN values since the Keras models can't handle that. The data is also split into training (80%) and test data (20%).
The type of neural network used is a Multi Layer Perceptron (MLP) network. The first layer is for normalization and centralisation, which means subtracting the mean and dividing with the variance for all values. This is because we don't want features that have naturally higher numbers to automatically get "stronger" weights.

The activation function used is ReLU, which is the best option since we use hidden layers in the model. With hiddens layers the ReLu function among other things helps to avoid the issue of vanishing gradients. For the last layer the activation function is linear instead of ReLu since we want a regression line as output. We use a dropout rate of 0.2 to minimize the risk of weights co-adapting. **For each hidden layer a regularization layer is added that allows penalties to be applied to layer paramters and activity. A manual grid search was performed testing different values for L1 and L2. L1 (0, 1e-6, 1e-5, 1e-4, 1e-3) ) and L2 ({0, 1e-6, 1e-5, 1e-4, 1e-3) was all tested and the best combination chosen.** Early stopping is added as a regularization method to prevent overfitting of the model. ADAM is used as an optimizer since its one of the more advanced ones, using both the methods of Momentum and RSMProp. Which means that the ADAM optimizer add a bit of the previous gradient to the current one, and adjusts the learning reate for each parameter based on the size of the recent gradients, and then also individually adjusts for each parameter.

In [ ]:
# Create a copy to avoid SettingWithCopyWarning
CHEMBL_df_final = CHEMBL_df_processed.copy()

# Select columns to use for training the model, can only be numeric
columns_to_use = ['Molecular Weight', 'AlogP']

# Remove NaN values from the relevant numerical columns before splitting
for col in columns_to_use + ['pChEMBL Value']:
    CHEMBL_df_final[col] = CHEMBL_df_final[col].replace([np.inf, -np.inf], np.nan)

CHEMBL_df_final.dropna(subset=columns_to_use + ['pChEMBL Value'], inplace=True)

# Extract the numerical features and the fingerprints
numerical_features = CHEMBL_df_final[columns_to_use]
fingerprints = pd.DataFrame(CHEMBL_df_final['Fingerprint'].tolist(), index=CHEMBL_df_final.index)

# Combine numerical features and fingerprints
x = pd.concat([numerical_features, fingerprints], axis=1)
y = CHEMBL_df_final['pChEMBL Value']

# Split into training and test data
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=12)

# Make the splits numpy arrays, input for keras
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [ ]:
# Set modelling parameter
epochs = 200    # Total throughput

# Define and adapt normalization layer
normalizer = Normalization(input_shape=(X_train.shape[1],)) # Use the actual number of features
normalizer.adapt(X_train)  # computes mean and variance from the data

# Create a Sequential model, each layer is stacked in order, each layers output is the next layers input
binding_model = Sequential([
    # Normalization (subtract the average) and centralization (divide by variance) layer
    normalizer,
    # Three hidden layers
    Dense(256, activation='relu'),
    Dropout(0.2),
    ActivityRegularization(l1=0.0, l2=1e-6),
    Dense(128, activation='relu'),
    Dropout(0.2),
    ActivityRegularization(l1=0.0, l2=1e-6),
    Dense(64, activation='relu'),
    Dropout(0.2),
    ActivityRegularization(l1=0.0, l2=1e-6),
    # Output layer only has one neuron since output is a regression line
    Dense(1, activation='linear')
])


early_stop = EarlyStopping(
    monitor='val_mae',     # watch validation MAE
    patience=20,           # stop after 20 epochs without improvement
    restore_best_weights=True
)

# Since data is tabular, Flatten is not needed
binding_model.compile(loss='mse', optimizer=keras.optimizers.Adam(),metrics=['mae'])
binding_model.summary()

binding_train = binding_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=epochs,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# Plot overfitting
plt.plot(binding_train.history['mae'], label='Train MAE')
plt.plot(binding_train.history['val_mae'], label='Val MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()
plt.show()

In [ ]:
# Predicted vs Actual Scatter Plot
# x-axis: actual pChEMBL values (true binding strength).
# y-axis: predicted values.

plt.scatter(y_test, binding_model.predict(X_test), alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)],
         linestyle='--', color='red', label='Perfect Prediction')
plt.xlabel('Actual pChEMBL Value')
plt.ylabel('Predicted pChEMBL Value')
plt.title('Actual vs. Predicted pChEMBL Values')
plt.legend()
plt.show()

In [ ]:
# Make predictions of two specific molecules just to make it easier to visualize results

# Generate 2 random numbers
nr_1, nr_2 = np.random.choice(len(X_test), size=2, replace=False)

# Make prediction for the first value
sample_1 = X_test[nr_1].reshape(1, -1)
actual_value_1 = y_test[nr_1]

#Predict
predicted_value_1 = binding_model.predict(sample_1)
print(f"Actual: {actual_value_1:.3f}, Predicted: {predicted_value_1[0][0]:.3f}")

# Make prediction for second value
sample_2 = X_test[nr_2].reshape(1, -1)
actual_value_2 = y_test[nr_2]

#Predict
predicted_value_2 = binding_model.predict(sample_2)
print(f"Actual: {actual_value_2:.3f}, Predicted: {predicted_value_2[0][0]:.3f}")

In [ ]:
# Summarize performace
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = binding_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MAE: {mae:.3f}")
print(f"R² Score: {r2:.3f}")

# Reflection
This model is, as stated previously, not very useable in its current state. It does, however, work as a blueprint for more advanced models. Ideally the model would have been trained to an extent that any inputted fingerprint would yield an expected binding affinity. Other possible improvements would be to add some form of indexing function where the analyzed fingerprints could be 'backwards translated' to provide the structure of the tested molecules. An interesting follow up would be to do some clustering on the output to perhaps identify which molecular qualities contribute the most to high binding affinity.

Finally, to make the model adhere better to the fair principles the program should ideally take the data from the website without the need to download it locally. The reason this was not easily implemented is that the molecular data was stored in a large array that took a very long time to loop through.

# Reflection relating to the FAIR priciples

**Findable:**
Since this document is a private colab notebook it can only be viewed with people it is shared with, when shared however it is findable. To improve the "findability" one could upload it on github and/or have a DOI.

**Accessible:**
Google colab is a very accessible way to share code, the code has headers so it is easy to follow the code, it's commented and written in plain english. Steps and decisions are explained.

**Interoperable:**
Python is used wich is the most commonly used coding language amd making additions to this code would be fairly easy.

**Reusable:**
One way the notebook is resusable is that the way to find the input data is described carefully insted of just provided. In my experience when the input data is is shared with a link it is common that it stops working after a while. However, a even better way would of course be to upload the notebook to GitHub as well as the input file.